## High quality T and B cell markers using pseudobulk DE on data with biological replicates
We follow the DE best practices tutorial (https://www.sc-best-practices.org/conditions/differential_gene_expression.html) but focus on T vs B cell DE on the control group only.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

import numpy as np
import pandas as pd
import pertpy
import scanpy as sc
import scipy.sparse as sp
import random

import rpy2.rinterface_lib.callbacks
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri, numpy2ri

import anndata2ri

%load_ext rpy2.ipython

In [2]:
%%R
library(edgeR)

Loading required package: limma


In [3]:
adata = pertpy.data.kang_2018()
adata

AnnData object with n_obs × n_vars = 24673 × 15706
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name'
    obsm: 'X_pca', 'X_umap'

In [4]:
adata.obs['label'].value_counts()

label
stim    12358
ctrl    12315
Name: count, dtype: int64

In [5]:
adata.obs["cell_type"].value_counts()

cell_type
CD4 T cells          11238
CD14+ Monocytes       5697
B cells               2651
NK cells              1716
CD8 T cells           1621
FCGR3A+ Monocytes     1089
Dendritic cells        529
Megakaryocytes         132
Name: count, dtype: int64

In [6]:
# Keep only T and B cells from control group
adata = adata[adata.obs["label"] == "ctrl", :]
adata
# # Keep only T and B cells
adata = adata[adata.obs["cell_type"].isin(["CD4 T cells", "CD8 T cells", "B cells"]), :]# adata.obs["cell_type_coarse"] = adata.obs["cell_type"]
adata.obs["cell_type"] = adata.obs["cell_type"].astype(str).replace(
    {"CD4 T cells": "T cells", "CD8 T cells": "T cells"}
)
adata.obs["cell_type"].value_counts()


cell_type
T cells    6371
B cells    1316
Name: count, dtype: int64

In [7]:
import scanpy as sc
adata.layers["counts"] = adata.X.copy()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata

AnnData object with n_obs × n_vars = 7687 × 13140
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters', 'n_genes'
    var: 'name', 'n_cells'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'

In [8]:
adata.obs["sample"] = [
    f"{rep}_{l}" for rep, l in zip(adata.obs["replicate"], adata.obs["label"])
]

adata.obs["cell_type"] = [ct.replace(" ", "_") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace("+", "") for ct in adata.obs["cell_type"]]

adata.obs["replicate"] = adata.obs["replicate"].astype("category")
adata.obs["label"] = adata.obs["label"].astype("category")
adata.obs["sample"] = adata.obs["sample"].astype("category")
adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")

In [9]:
NUM_OF_CELL_PER_DONOR = 30


def aggregate_and_filter(
    adata,
    cell_identity,
    donor_key="sample",
    condition_key="label",
    cell_identity_key="cell_type",
    obs_to_keep=None,  # which additional metadata to keep, e.g. gender, age, etc.
    replicates_per_patient=1,
):
    # subset adata to the given cell identity
    if obs_to_keep is None:
        obs_to_keep = []
    adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
    # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
    size_by_donor = adata_cell_pop.obs.groupby([donor_key]).size()
    donors_to_drop = [
        donor
        for donor in size_by_donor.index
        if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
    ]
    if len(donors_to_drop) > 0:
        print("Dropping the following samples:")
        print(donors_to_drop)
    df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

    adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
    for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
        print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
        if donor not in donors_to_drop:
            adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
            # create replicates for each donor
            indices = list(adata_donor.obs_names)
            random.shuffle(indices)
            indices = np.array_split(np.array(indices), replicates_per_patient)
            for i, rep_idx in enumerate(indices):
                adata_replicate = adata_donor[rep_idx]
                # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
                agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
                for obs in obs_to_keep:
                    agg_dict[obs] = "first"
                # create a df with all genes, donor and condition info
                X = adata_replicate.X
                arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
                df_donor = pd.DataFrame(arr)
                df_donor.index = adata_replicate.obs_names
                df_donor.columns = adata_replicate.var_names
                df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
                # aggregate
                df_donor = df_donor.groupby(donor_key).agg(agg_dict)
                df_donor[donor_key] = donor
                df.loc[f"donor_{donor}_{i}"] = df_donor.loc[donor]
    print("\n")
    # create AnnData object from the df
    adata_cell_pop = sc.AnnData(
        df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)
    )
    return adata_cell_pop

In [10]:
%%R
fit_model <- function(adata_){
    # create an edgeR object with counts and cell type as the grouping factor
    y <- DGEList(assay(adata_, "X"), group = colData(adata_)$cell_type)
    # filter out genes with low counts
    print("Dimensions before subsetting:")
    print(dim(y))
    print("")
    keep <- filterByExpr(y, group = colData(adata_)$cell_type)
    y <- y[keep, , keep.lib.sizes=FALSE]
    print("Dimensions after subsetting:")
    print(dim(y))
    print("")
    # normalize
    y <- calcNormFactors(y)
    # Use cell type grouping variable for DE
    group <- colData(adata_)$cell_type
    replicate <- colData(adata_)$replicate
    # design matrix: test for DE between cell types while accounting for donor replicates
    design <- model.matrix(~ 0 + group + replicate)
    # estimate dispersion
    y <- estimateDisp(y, design = design)
    # fit the model
    fit <- glmQLFit(y, design)
    return(list("fit"=fit, "design"=design, "y"=y))
}

In [11]:
obs_to_keep = ["label", "cell_type", "replicate", "sample"]


In [12]:
adata.X = adata.layers["counts"].copy()


In [13]:
# process first cell type separately...
cell_type = adata.obs["cell_type"].cat.categories[0]
print(
    f'Processing {cell_type} (1 out of {len(adata.obs["cell_type"].cat.categories)})...'
)
adata_pb = aggregate_and_filter(adata, cell_type, obs_to_keep=obs_to_keep)
for i, cell_type in enumerate(adata.obs["cell_type"].cat.categories[1:]):
    print(
        f'Processing {cell_type} ({i+2} out of {len(adata.obs["cell_type"].cat.categories)})...'
    )
    adata_cell_type = aggregate_and_filter(adata, cell_type, obs_to_keep=obs_to_keep)
    adata_pb = adata_pb.concatenate(adata_cell_type)

Processing B_cells (1 out of 2)...
Dropping the following samples:
['patient_1039_ctrl']
	Processing donor 8 out of 8...

Processing T_cells (2 out of 2)...
	Processing donor 8 out of 8...



In [14]:
adata_pb.layers['counts'] = adata_pb.X.copy()


In [15]:
sc.pp.normalize_total(adata_pb, target_sum=1e6)
sc.pp.log1p(adata_pb)
sc.pp.pca(adata_pb)

adata_pb.obs["lib_size"] = np.sum(adata_pb.layers["counts"], axis=1)
adata_pb.obs["log_lib_size"] = np.log(adata_pb.obs["lib_size"].astype(float))

adata_pb.X = adata_pb.layers['counts'].copy()


In [16]:
counts = adata_pb.X
if sp.issparse(counts):
    counts = counts.T.tocsc()   # genes x cells for edgeR, CSC for dgCMatrix
else:
    counts = np.asarray(counts).T

meta = adata_pb.obs[["cell_type", "replicate"]].copy()

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["meta"] = ro.conversion.py2rpy(meta)

# sparse dgCMatrix payload
if sp.issparse(counts):
    csc = counts
    ro.globalenv["i"] = ro.IntVector(csc.indices.astype(np.int32))
    ro.globalenv["p"] = ro.IntVector(csc.indptr.astype(np.int32))
    ro.globalenv["x"] = ro.FloatVector(csc.data.astype(np.float64))
    ro.globalenv["dims"] = ro.IntVector(list(map(int, csc.shape)))
    ro.globalenv["gene_names"] = ro.StrVector(list(map(str, adata_pb.var_names)))
    ro.globalenv["cell_names"] = ro.StrVector(list(map(str, adata_pb.obs_names)))
else:
    mat = np.asarray(counts, dtype=np.float64, order="F")
    ro.globalenv["counts_dense"] = ro.r.matrix(ro.FloatVector(mat.ravel("F")),
                                              nrow=mat.shape[0], ncol=mat.shape[1])
    ro.globalenv["gene_names"] = ro.StrVector(list(map(str, adata_pb.var_names)))
    ro.globalenv["cell_names"] = ro.StrVector(list(map(str, adata_pb.obs_names)))


In [17]:
%%R
suppressPackageStartupMessages({
  library(Matrix)
  library(SingleCellExperiment)
  library(edgeR)
})

if (exists("counts_dense")) {
  counts <- counts_dense
} else {
  counts <- new("dgCMatrix", i=i, p=p, x=x, Dim=dims)
}
rownames(counts) <- gene_names
colnames(counts) <- cell_names

sce <- SingleCellExperiment(list(X = counts), colData = meta)
outs <- fit_model(sce)

[1] "Dimensions before subsetting:"
[1] 13140    15
[1] ""
[1] "Dimensions after subsetting:"
[1] 3095   15
[1] ""


In [18]:
%%R
fit <- outs$fit
y <- outs$y

In [19]:
%%R
colnames(y$design)

[1] "groupB_cells"          "groupT_cells"          "replicatepatient_1015"
[4] "replicatepatient_1016" "replicatepatient_1039" "replicatepatient_107" 
[7] "replicatepatient_1244" "replicatepatient_1256" "replicatepatient_1488"


In [20]:
%%R -o tt_TvB
myContrast_TvB <- limma::makeContrasts(groupT_cells - groupB_cells, levels = colnames(y$design))
qlf_TvB <- glmQLFTest(fit, contrast = myContrast_TvB)
tt_TvB <- topTags(qlf_TvB, n = Inf)$table

In [21]:
%%R -o tt_BvT
myContrast_BvT <- limma::makeContrasts(groupB_cells - groupT_cells, levels = colnames(y$design))
qlf_BvT <- glmQLFTest(fit, contrast = myContrast_BvT)
tt_BvT <- topTags(qlf_BvT, n = Inf)$table



In [22]:
%%R
T_markers <- tt_TvB[tt_TvB$FDR < 0.05 & tt_TvB$logFC > 0, ]
B_markers <- tt_TvB[tt_TvB$FDR < 0.05 & tt_TvB$logFC < 0, ]

In [23]:
tt_TvB = %R tt_TvB
T_markers = %R T_markers
B_markers = %R B_markers

In [24]:
T_markers.to_csv("T_markers.csv")
B_markers.to_csv("B_markers.csv")

In [25]:
T_markers.head(20)

,logFC,logCPM,F,PValue,FDR
CD3E,4.610397,8.132056,758.977045,1.272226e-16,6.562564e-14
GIMAP5,5.275619,8.010078,523.893489,7.212962e-16,2.499254e-13
NKG7,5.427481,8.135846,447.437117,1.049542e-15,2.953030e-13
IL32,5.373878,7.938645,437.816821,1.850215e-15,4.772014e-13
CD2,5.214173,8.738835,558.378670,2.107290e-15,4.906123e-13
CD3D,5.078476,9.391334,659.786064,2.219248e-15,4.906123e-13
LEPROTL1,3.412030,8.699013,680.363954,2.854710e-15,5.890218e-13
CD7,5.293567,8.588565,503.192328,3.354251e-15,6.488379e-13
AMICA1,5.622711,7.015547,309.898400,4.112329e-15,7.486858e-13
GNLY,4.670921,8.302299,451.046216,5.587029e-15,9.100977e-13


In [26]:
B_markers.head(20)

,logFC,logCPM,F,PValue,FDR
CD74,-4.625854,12.369884,2372.319578,7.817185e-20,2.419419e-16
IGJ,-7.721372,6.205478,738.729292,8.143458e-18,1.260200e-14
LINC00926,-8.309192,5.727980,595.676844,1.643420e-17,1.695462e-14
CD79B,-5.717297,7.297527,1126.863739,2.345101e-17,1.814522e-14
AL928768.3,-7.327640,5.189683,442.407077,9.457361e-17,5.854106e-14
CD79A,-6.527561,9.470414,854.804887,4.407985e-16,1.948959e-13
IGLL5,-10.061497,7.282605,949.563240,7.267621e-16,2.499254e-13
ADAM28,-6.550867,5.746166,429.979760,8.901057e-16,2.754877e-13
IRF8,-5.379550,6.669951,525.002895,4.748696e-15,8.165120e-13
ACP5,-4.758092,6.372515,616.900051,6.572635e-15,1.017115e-12
